In [1]:
import os, re
import pandas as pd

In [2]:
# select event type
event = 'heat_wave'
#event = 'cold_snap'

# set input and output paths
path_input = os.path.join('..', 'Data', 'extreme_data_raw', f'{event}_library_NERC_average_pop_def9.csv')
path_output = os.path.join('..', 'Data', 'extreme_data_clean', f'{event}_impact_timeseries.csv')

# read input file
df_input = pd.read_csv(path_input)

# calculate impact = spatial_coverage * duration * (extr_temp - 290) for heat wave, spatial_coverage * duration * (290 - extr_temp) for cold snap
if 'highest_temperature' in df_input.columns: # heat wave
    df_input = df_input.rename(columns = {'highest_temperature': 'extr_temp'})
    df_input['impact'] = df_input['spatial_coverage'] * df_input['duration'] * (df_input['extr_temp'] - 290) # extr_temp > 290K
elif 'lowest_temperature' in df_input.columns: # cold snap
    df_input = df_input.rename(columns = {'lowest_temperature': 'extr_temp'})
    df_input['impact'] = df_input['spatial_coverage'] * df_input['duration'] * (290 - df_input['extr_temp']) # extr_temp < 290K

# print
df_input.head()

,start_date,end_date,centroid_date,extr_temp,duration,NERC_ID,spatial_coverage,impact
0,1980-05-29,1980-05-30,1980-05-29,299.531094,2,NERC18,38.396624,731.923709
1,1980-06-13,1980-06-14,1980-06-14,301.741869,2,NERC18,43.881857,1030.510056
2,1980-06-23,1980-06-27,1980-06-26,305.284515,5,NERC18,86.919831,6642.637245
3,1980-07-07,1980-07-16,1980-07-11,308.081620,10,NERC18,95.147679,17204.241901
4,1980-07-18,1980-07-20,1980-07-19,304.800949,3,NERC18,57.172996,2538.643700


In [3]:
df_summary = pd.DataFrame()

# sort NERC_ID in natural order
nerc_id = sorted(df_input['NERC_ID'].unique(), key = lambda s: [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', s)])
# iterate through each NERC_ID
for idx in nerc_id:
    # select rows with NERC_ID
    df_sel = df_input[df_input['NERC_ID'] == idx]

    # spread events into timeseries with start_date, end_date, and impact
    df_sel_spread = df_sel.apply(lambda x: pd.Series(x['impact'], index = pd.date_range(x['start_date'], x['end_date'], freq = 'D')), axis = 1).stack().dropna().to_frame(idx)
    df_sel_spread = df_sel_spread.reset_index(level = 0, drop = True)
    df_sel_spread.index.name = 'date'
    
    # merge with summary dataframe
    df_summary = df_summary.merge(df_sel_spread, left_index = True, right_index = True, how = 'outer')

# fill NaN with -1 (no event)
df_summary = df_summary.fillna(-1)

# round to 2 decimal places and save to csv
df_summary = df_summary.round(2)

# save to csv
#df_summary.to_csv(path_output)

# print
df_summary.head()

,NERC1,NERC2,NERC3,NERC4,NERC5,NERC6,NERC7,NERC8,NERC9,NERC10,NERC11,NERC12,NERC15,NERC17,NERC18,NERC20
date,,,,,,,,,,,,,,,,
1980-05-29,-1.0,-1.0,-1.00,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,731.92,-1.00
1980-05-30,-1.0,-1.0,-1.00,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,731.92,-1.00
1980-06-05,-1.0,-1.0,-1.00,-1.0,-1.0,-1.0,-1.0,3826.9,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.00,-1.00
1980-06-06,-1.0,-1.0,-1.00,-1.0,-1.0,-1.0,-1.0,3826.9,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.00,3243.68
1980-06-07,-1.0,-1.0,1176.84,-1.0,-1.0,-1.0,-1.0,3826.9,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.00,3243.68
